In [105]:
!pip install cellpose

In [106]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q

In [107]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import tifffile as tiff
from cellpose import plot


class CalciumRecording:
    def __init__(self, path: str):
        self.data: np.ndarray = self._load(path)
        print(self.data.shape)
        self.path = path

    @staticmethod
    def _load_avi(path: str):
        cap = cv2.VideoCapture(path)

        frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(frame)

        cap.release()
        return np.array(frames)

    def _load(self, path: str):
        if "avi" in path:
            data = CalciumRecording._load_avi(path)
        else:
            data = tiff.imread(path)

        return data

    def visualize(self, masks: np.ndarray, flows: np.ndarray):
        from matplotlib.animation import FuncAnimation, FFMpegWriter
    
        fig = plt.figure(figsize=(20, 20))
    
        def update(t):
            fig.clf()
            plot.show_segmentation(fig, self.data[t], masks[t], flows[t][0])
    
        ani = FuncAnimation(fig, update, frames=self.data.shape[0], interval=100)
        writer = FFMpegWriter(fps=10)
        ani.save("segmentation.mp4", writer=writer)
        plt.close()
        print("Saved to segmentation.mp4")



In [108]:
from cellpose import models


class Segmenter:
    def __init__(self):
        pass

    @staticmethod
    def generate_mask(recording: CalciumRecording):
        model = models.CellposeModel(gpu=True)
        data = [recording.data[i] for i in range(recording.data.shape[0])]
        masks, flows, styles = model.eval(data, diameter=None, channels=[0, 0])
        return masks, flows


In [109]:
import pickle
import numpy as np
import os

In [110]:
from pathlib import Path

file_path = "/kaggle/input/datasets/mokarbalaee/neuroseg/Medien2.avi"
file_name = Path(file_path).name
recording: CalciumRecording = CalciumRecording(path=file_path)

(2160, 200, 214, 3)


In [111]:
def save_results(masks, flows):
    np.save(f"masks+{file_name}.npy", masks)
    with open(f"flows+{file_name}.pkl", "wb") as f:
        pickle.dump(flows, f)

def load_results():
    masks = np.load(f"masks+{file_name}.npy", allow_pickle=True)
    with open(f"flows+{file_name}.pkl", "rb") as f:
        flows = pickle.load(f)
    return masks, flows

In [112]:
if os.path.exists(f"masks+{file_name}.npy"):
    masks, flows = load_results()
else:
    masks, flows = Segmenter.generate_mask(recording)
    save_results(masks, flows)


print(len(masks))
print(len(flows))

2160
2160


In [ ]:
recording.visualize(masks, flows)

In [ ]:
from IPython.display import Video
Video("segmentation.mp4")